# Self-Supervised Learning of Spatial Transcriptomics


This notebook demonstrates how to train a self supervised model on a collection of anndata objects and produce a "complete" checkpoint file.  

Here we are going to train the model for a short time just for demonstration.  

To reproduce the results in the paper run the scripts in the "run" folder.  

See documentation for more details.

## Common imports

In [ ]:
import numpy
import torch
import seaborn
import tarfile
import os
import matplotlib
import matplotlib.pyplot as plt
from anndata import read_h5ad

# import tissue mosaic
import tissuemosaic as tm

In [ ]:
## set seeds
import random
import numpy as np

r_seed=t_seed=n_seed=100

random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.trainer import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from tissuemosaic.models import NeptuneLoggerCkpt
from tissuemosaic.data import AnndataFolderDM
from tissuemosaic.plots import show_raw_all_channels, show_raw_one_channel


## Read in and preprocess anndata

In [ ]:
# Replace this with the path to where you have downloaded the data
data_destination_folder = os.path.abspath('../../TissueMosaic_Manuscript/TissueMosaic_data/testis_anndata/')

# Make a list of all the h5ad files in the data_destination_folder
fname_list = []
for f in os.listdir(data_destination_folder):
    if f.endswith('.h5ad'):
        fname_list.append(f)
print(fname_list)

In [ ]:
# read in all the anndata

anndata_list = []
for fname in fname_list:
    try:
        anndata = read_h5ad(os.path.join(data_destination_folder, fname))
        print("Loaded {}".format(fname))
        anndata_list.append(anndata)
    except:
        pass

In [ ]:
for i,anndata in enumerate(anndata_list):
    anndata.obs['cell_type'] = anndata.obsm['cell_type_proportions'].idxmax(axis=1)
    anndata.write_h5ad(os.path.join(data_destination_folder, fname_list[i]))

## Visualize anndata as a sanity check

In [ ]:
anndata_list[0]

In [ ]:
## Plot cell types in space

ncols=3
nrows=2
fig, axes = plt.subplots(ncols=ncols, nrows=nrows, figsize=(6*ncols,6*nrows))

# Define an consistent mapping from cell_type and color for all the plots
cell_types = numpy.asarray(anndata_list[0].obs['cell_type'].values)
unique_cell_types = numpy.unique(cell_types)

n = -1
for r in range(nrows):
    for c in range(ncols):
        n += 1
        anndata_tmp = anndata_list[n]
        cell_types = numpy.asarray(anndata_tmp.obs['cell_type'].values)
        x = numpy.asarray(anndata_tmp.obs['x'].values)
        y = numpy.asarray(anndata_tmp.obs['y'].values) 
        seaborn.scatterplot(x=x, y=y, hue=cell_types, ax=axes[r,c], size=numpy.ones_like(x), sizes=(10, 10), hue_order=unique_cell_types) 
        _ = axes[r,c].set_title(fname_list[n])

In [ ]:
## Plot cell type distribution in each sample

x_labels_rotation = 90
fig, axes = plt.subplots(ncols=ncols, nrows=nrows, figsize=(6*ncols,6*nrows))
n = -1
for r in range(nrows):
    for c in range(ncols):
        n += 1
        counts = anndata_list[n].obs["cell_type"].value_counts(sort=False)
        x = numpy.asarray(counts.index)
        y = counts.to_numpy()
        _ = seaborn.barplot(x=x, y=y, ax=axes[r,c])
        x_labels_raw = axes[r,c].get_xticklabels()
        axes[r,c].set_xticklabels(labels=x_labels_raw, rotation=x_labels_rotation)
        _ = axes[r,c].set_title(fname_list[n])

## Instantiate DataModule 

Here we use the defaults parameters for the datamodule and only define:

1. The mapping from cell_type to channels in the image. Each channel will represent the density of a specific cell_type. In some situation it may make sense to map multiple cell-types to the same channel. For example CD4+ and CD4- cells might be mapped to the same channel.
2. The folder with the anndata h5ad files

In [ ]:
# Load config from YAML file first to get the correct categories_to_channels
import yaml
with open('../run/config_dino_ssl_testis.yaml', 'r') as f:
    config_yaml = yaml.safe_load(f)

# Use the categories_to_channels from the YAML config 
categories_to_channels = config_yaml['categories_to_channels'].copy()

config_dm = tm.data.AnndataFolderDM.get_default_params() # get the defaults parameters
config_dm["data_folder"] = data_destination_folder  # specify the folder with the anndata h5ad files
config_dm["categories_to_channels"] = categories_to_channels  # specify the mapping between cell_types and channels
config_dm['category_key'] = 'cell_type_proportions' # specify which key in obsm contains input features

dm = tm.data.AnndataFolderDM(**config_dm)

## Instantiate the Model

Here we use Dino, but the same approach works for Barlow Twins, SimCLR, and VAE.

We use the defaults parameters and only change the number of input channels of the image.

In [ ]:
from tissuemosaic.models.ssl_models import *
# now you can access: Barlow, Simclr, Dino, Vae

config_model = tm.models.ssl_models.Dino.get_default_params()  # get the default parameters

# Use the already loaded config_yaml from the previous cell
for key in config_yaml:
    config_model[key] = config_yaml[key]

config_model['image_in_ch'] = dm.ch_in  # specify the number of input channels consistently with datamodule

config_model.update(config_dm)  # concatenate the two configuration dictionaries
model = tm.models.ssl_models.Dino(**config_model)  
# Now the checkpoint contains the full information to reproduce the simulation.

## Train the model and save the final checkpoint

We use Neptune to log our results.  

Here Neptune is run in the 'offline' mode and the logs are written to local disk.  

Neptune can run in 'async' mode and the results will be saved on a remote database with a nice graphic interface.  

For the 'async' option to work you need to sign up for a free account and provide the correct project and api_key.  

See https://docs.neptune.ai/ for more info.

In [ ]:
pl_neptune_logger = NeptuneLoggerCkpt(
    api_key="ANONYMOUS",  # replace with your own
    project='cellarium/tissue-purifier', # replace with your own
    run=None,  # if None a new run will be logged. If the run is provided the result will be appended to existing run  
    log_model_checkpoints=True, 
    mode="offline",  # "async"
    tags=["test"],
    fail_on_exception=True,  
)

# Save the checkpoint periodically during training
ckpt_train = ModelCheckpoint(
    save_weights_only=False,
    save_on_train_epoch_end=True,
    save_last=True,
    every_n_epochs=5,
)
    
# Define the trainer
pl_trainer = Trainer(
    # weights_save_path="saved_ckpt",
    callbacks=[ckpt_train],
    gpus=1,#torch.cuda.device_count(),  # number of gpu cards on a single machine to use
    check_val_every_n_epoch=10,
    num_sanity_val_steps=0,
    max_epochs=5, #config_model["max_epochs"],  # run for a 5 epochs for demonstration
    logger=pl_neptune_logger,
    log_every_n_steps=100,
    sync_batchnorm=True,
    accelerator="auto",
    devices=1
)

In [ ]:
# Fit the model to the data. 
# To obtain the best results, run till loss has converged (increase 'max_epochs' in the previous cell). 
# This will typically be easier to do from command line, following run/main_1_train_ssl.py
pl_trainer.fit(model=model, datamodule=dm)

## Save checkpoint

Since the model was instantiate using a dictionary containing all the parameters (both for model and datamodule) the checkpoint is complete.  

A single checkpoint file contains all the information needded to reproduce the simulation.

In [ ]:
pl_trainer.save_checkpoint("ckpt_dino_ssl.pt")  

## Visualize the crops used for training as a sanity check

In [ ]:
train_loader = dm.train_dataloader()  # get the train_dataloader from the datamodule
batch = next(iter(train_loader))  # get one batch from the dataloader
list_sp_imgs, list_labels, list_metadata = batch  # batch consists of 3 lists: sparse_images, labels, metadata

In [ ]:
n_examples = 5  # number of distinct crops
n_augmentations = 3  # apply the random data augmentation this many times

all_imgs = []
for n in range(n_augmentations):
    imgs_tmp = dm.trsfm_train_global(list_sp_imgs[:n_examples])  # apply the data augmentations
    all_imgs.append(imgs_tmp)
    
imgs_train = torch.cat(all_imgs, dim=0)
print("imgs_train.shape ->", imgs_train.shape)

In [ ]:
# Each column is a different patch
# Each row is a different instance of the random data-augmentation

titles = []
for r in range(n_augmentations):
    for c in range(n_examples):
        titles.append("crop = {}, augmentation = {}".format(c,r))

train_all_ch_fig = show_raw_all_channels(imgs_train, 
                                         cmap="viridis", 
                                         n_col=n_examples, 
                                         figsize=(4*n_examples, 4*n_augmentations), 
                                         sup_title="Train crops, all channels",
                                        titles=titles)
train_all_ch_fig